# Фаза 1: сбор эпизодов на задачах MiMo с адаптером `pref_edit_v1` (Colab)

Модель с адаптером крутится здесь (vLLM), инструменты и проверка тестов идут на ноутбуке с Docker через туннель
(`python -m sim.mimo_server --data ... --prune-images`). Цель сбора: успешные эпизоды, пары для обучения
по другим дефектам `edit_file` (не совпал `old_string`) и статистика по десяткам репозиториев.

Порядок: **1** карта, **2** код, **3** Drive и модель, **4** vLLM с адаптером, **5** сервер песочницы,
**6** пробный прогон 25 задач (сначала только он), **7** его разбор, **8** большой прогон (после пробного).
Результаты большого прогона пишутся прямо в Drive: при обрыве сеанса запуск той же ячейки продолжит с места остановки.

## 1. Карта (подойдёт A100 40 или 80 ГБ)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 2. Код и зависимости

In [ ]:
import os
if not os.path.exists("/content/gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git /content/gemma4-swe-rl
%cd /content/gemma4-swe-rl
!git pull -q
!git log --oneline | head -3
# torch не трогаем: его ставит vllm. Большая установка, только после git pull
!pip install -q vllm openai pyyaml requests huggingface_hub kagglehub
!python -c "import vllm; print('vllm', vllm.__version__)"

## 3. Drive, базовая модель и адаптер

In [ ]:
import getpass, os, shutil
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gemma_adapters"
ADAPTER = "pref_edit_v1"
MODEL_DIR = "/content/model"
HF_REPO = "google/gemma-4-31b-it-qat-w4a16-ct"
KAGGLE_MODEL = "google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct"

if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download(HF_REPO, local_dir=MODEL_DIR, token=hf_token)
        print("Скачано с Hugging Face")
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download(KAGGLE_MODEL)
        !cp -r "{path}"/. {MODEL_DIR}/
        print("Скачано с Kaggle Models")

os.makedirs("adapters", exist_ok=True)
if not os.path.exists(f"adapters/{ADAPTER}"):
    shutil.copytree(f"{DRIVE}/{ADAPTER}", f"adapters/{ADAPTER}")
print(os.listdir(f"adapters/{ADAPTER}"))

## 4. vLLM с адаптером
Параметры как в боевой среде (контекст 32768, парсеры `gemma4`, мышление включено). Если не хватает памяти на KV-кэш
(карта 40 ГБ), уменьшите `--max-model-len` до 24576.

In [ ]:
import subprocess, time, requests
cmd = ["vllm", "serve", MODEL_DIR, "--served-model-name", "gemma-4-31b-it-qat-w4a16-ct",
       "--quantization", "compressed-tensors", "--dtype", "bfloat16",
       "--max-model-len", "32768", "--gpu-memory-utilization", "0.92", "--max-num-seqs", "4",
       "--enable-lora", "--max-lora-rank", "64", "--max-loras", "1",
       "--lora-modules", f"{ADAPTER}=/content/gemma4-swe-rl/adapters/{ADAPTER}",
       "--enable-auto-tool-choice", "--tool-call-parser", "gemma4", "--reasoning-parser", "gemma4",
       "--default-chat-template-kwargs", '{"enable_thinking": true}', "--port", "8000"]
log = open("/content/vllm.log", "w")
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while time.time() - t0 < 1500:
    if proc.poll() is not None:
        break
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=3).status_code == 200:
            print("vLLM готов за %.0f c" % (time.time() - t0)); break
    except requests.RequestException:
        pass
    time.sleep(5)
if proc.poll() is not None:
    print("vLLM упал, конец лога:"); print(open("/content/vllm.log").read()[-3000:])
print([m["id"] for m in requests.get("http://127.0.0.1:8000/v1/models").json()["data"]])

## 5. Сервер песочницы (на ноутбуке)
Сначала запустите Docker Desktop и дождитесь, пока `docker ps` перестанет выдавать ошибку. Затем два окна PowerShell
из папки проекта:

```
python -m sim.mimo_server --data C:\Users\vishn\Desktop\data\mimo_v2_6_code --max-sessions 4 --ttl 2400 --prune-images
cloudflared tunnel --url http://127.0.0.1:8765
```
Сервер должен напечатать `Задач (python, pytest): 530`, токен и адрес `http://127.0.0.1:8765`; туннель печатает
`https://....trycloudflare.com`. `--prune-images` удаляет образ задачи после эпизода (образы по 1-7 ГБ).
Ноутбук не должен засыпать, питание подключено.

In [ ]:
import getpass, json, urllib.request
os.environ["SWE_SANDBOX_URL"] = input("Адрес туннеля: ").strip().rstrip("/")
os.environ["SWE_SANDBOX_TOKEN"] = getpass.getpass("Токен сервера: ").strip()
print(urllib.request.urlopen(os.environ["SWE_SANDBOX_URL"] + "/health", timeout=30).read())
req = urllib.request.Request(os.environ["SWE_SANDBOX_URL"] + "/tasks",
                             headers={"Authorization": "Bearer " + os.environ["SWE_SANDBOX_TOKEN"]})
ids = json.load(urllib.request.urlopen(req, timeout=30))["result"]
print(len(ids), "задач, примеры:", ids[:3])
assert len(ids) > 300, "ожидалось ~530 задач MiMo: сервер запущен с --data?"

## 6. Пробный прогон: 25 случайных задач (около 1-2 ч)
Цель пробного прогона: убедиться, что образы скачиваются и удаляются, посмотреть время «старт» (скачивание образа),
долю решённых и сколько вызовов `edit_file` с ошибкой набирается на эпизод. Зерно перемешивания 1; без системного промпта
(как в прежних сборах), температура 0.7, 4 потока.

In [ ]:
run_cmd = ("python -m sim.run_mimo_baseline --model " + ADAPTER + " --temperature 0.7 --repeats 1 --workers 4 "
           "--thinking on --no-analyzer --clean-start --first 25 --shuffle-seed 1 --out results/mimo_p1_pilot")
!{run_cmd}

## 7. Разбор пробного прогона

In [ ]:
!python -m tools.analyze_edits results/mimo_p1_pilot
import json
last = {}
for l in open("results/mimo_p1_pilot/results.jsonl", encoding="utf-8"):
    r = json.loads(l)
    last[(r["instance_id"], r.get("rep", 0))] = r
rows = list(last.values())
print("решено:", sum(1 for r in rows if r["passed"]), "из", len(rows), "| ошибок инфраструктуры:", sum(1 for r in rows if r.get("error")))
!zip -qr {DRIVE}/mimo_p1_pilot.zip results/mimo_p1_pilot && echo "копия в Drive"

## 8. Большой прогон (только после разбора пробного)
Все 530 задач в перемешанном порядке, результаты прямо в Drive. Прогон возобновляемый: после обрыва сеанса
повторите ячейки 2-5 и эту ячейку, уже готовые эпизоды будут пропущены. Остановить можно в любой момент (кнопка Stop),
часть данных всё равно будет разнообразной благодаря перемешиванию.

In [ ]:
OUT = f"{DRIVE}/mimo_p1_full"
run_cmd = ("python -m sim.run_mimo_baseline --model " + ADAPTER + " --temperature 0.7 --repeats 1 --workers 4 "
           "--thinking on --no-analyzer --clean-start --first 530 --shuffle-seed 1 --out " + OUT)
!{run_cmd}